<a href="https://colab.research.google.com/github/miguelvelascop/TFM_Miguel_Velasco_Puig/blob/main/Implementacion_NCR_Miguel_Velasco_Puig.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementación de Nuclear Chain Reaction (NCR)

Este notebook contiene la implementación del algoritmo Nuclear Chain Reaction, los doce problemas benchmark de ingeniería, la selección de parámetros y la evaluación final mediante 30 ejecuciones de cada problema.

> La última celda ejecuta el experimento completo y puede tardar bastante tiempo.

## 1. Importaciones, estructuras de datos y algoritmo NCR

In [1]:
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Sequence, Tuple, Any
import math
import time
from pathlib import Path
import numpy as np
import pandas as pd

EPS = 1e-12
INVALID_SCORE = 1e100


@dataclass
class BenchmarkProblem:
    # Contiene la definición de un problema benchmark: límites, función objetivo, restricciones y
    # reparación opcional.
    name: str
    bounds: Sequence[Tuple[float, float]]
    objective: Callable[[np.ndarray], float]
    constraints: Callable[[np.ndarray], Sequence[float]]
    repair: Optional[Callable[[np.ndarray], np.ndarray]] = None
    known_best: Optional[float] = None


@dataclass
class NCRResult:
    # Agrupa el resultado final de una ejecución de NCR, incluyendo la mejor solución, factibilidad,
    # restricciones e historial.
    best_x: np.ndarray
    best_objective: float
    best_score: float
    best_violation: float
    feasible: bool
    constraints: np.ndarray
    evaluations: int
    iterations: int
    history: Dict[str, List[float]]


class NCR:
    # Implementa la metaheurística NCR para problemas de minimización restringida con ranking de
    # factibilidad.

    # Define y documenta la función `__init__`.
    def __init__(
        self,
        objective: Callable[[np.ndarray], float],
        bounds: Sequence[Tuple[float, float]],
        constraints: Optional[Callable[[np.ndarray], Sequence[float]]] = None,
        repair: Optional[Callable[[np.ndarray], np.ndarray]] = None,
        population_size: int = 30,
        k: int = 3,
        p_min: float = 0.20,
        p_max: float = 0.90,
        elitism_rate: float = 0.10,
        neighborhood_rate: float = 0.10,
        penalty_factor: float = 1e12,
        max_evaluations: int = 30000,
        seed: Optional[int] = None,
    ):
        # Inicializa el algoritmo, guarda los parámetros y prepara los límites del espacio de búsqueda.
        self.objective = objective
        self.constraints = constraints
        self.repair = repair

        self.bounds = np.asarray(bounds, dtype=float)
        self.lb = self.bounds[:, 0]
        self.ub = self.bounds[:, 1]
        self.dim = len(self.lb)

        self.population_size = int(population_size)
        self.k = int(k)
        self.p_min = float(p_min)
        self.p_max = float(p_max)
        self.elitism_rate = float(elitism_rate)
        self.neighborhood_rate = float(neighborhood_rate)
        self.penalty_factor = float(penalty_factor)
        self.max_evaluations = int(max_evaluations)
        self.rng = np.random.default_rng(seed)
        self.evaluations = 0

        self._validate_parameters()

    # Comprueba que los parámetros principales tengan valores válidos antes de ejecutar NCR.
    def _validate_parameters(self) -> None:
        if self.population_size <= 0:
            raise ValueError("population_size must be > 0")
        if self.k <= 0:
            raise ValueError("k must be > 0")
        if not (0.0 <= self.p_min < self.p_max <= 1.0):
            raise ValueError("p_min and p_max must satisfy 0 <= p_min < p_max <= 1")
        if not (0.0 <= self.elitism_rate < 1.0):
            raise ValueError("elitism_rate must be in [0, 1)")
        if not (0.0 < self.neighborhood_rate <= 1.0):
            raise ValueError("neighborhood_rate must be in (0, 1]")
        if self.max_evaluations < self.population_size:
            raise ValueError("max_evaluations must be at least population_size")
        if np.any(self.lb >= self.ub):
            raise ValueError("Each lower bound must be smaller than its upper bound")

    # Ajusta una solución a los límites del problema y aplica la reparación de variables enteras o discretas.
    def _clip_and_repair(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=float).copy()
        x = np.clip(x, self.lb, self.ub)
        if self.repair is not None:
            x = np.asarray(self.repair(x), dtype=float)
            x = np.clip(x, self.lb, self.ub)
        return x

    # Evalúa una solución individual calculando objetivo, restricciones, violación total, factibilidad y valor penalizado.
    def _evaluate_one(self, x: np.ndarray) -> Dict[str, Any]:
        x = self._clip_and_repair(x)
        self.evaluations += 1

        try:
            obj = float(self.objective(x))
        except Exception:
            obj = INVALID_SCORE

        if self.constraints is None:
            g = np.array([], dtype=float)
        else:
            try:
                g = np.asarray(self.constraints(x), dtype=float)
            except Exception:
                g = np.array([INVALID_SCORE], dtype=float)

        if not np.isfinite(obj):
            obj = INVALID_SCORE
        if g.size and not np.all(np.isfinite(g)):
            g = np.where(np.isfinite(g), g, INVALID_SCORE)

        # Todas las restricciones deben implementarse como g_j(x) <= 0.
        violation = float(np.sum(np.maximum(0.0, g) ** 2)) if g.size else 0.0
        if not np.isfinite(violation):
            violation = INVALID_SCORE

        score = obj + self.penalty_factor * violation
        if not np.isfinite(score):
            score = INVALID_SCORE

        feasible = bool(np.all(g <= 1e-8)) if g.size else True
        return {
            "x": x,
            "objective": obj,
            "score": score,
            "violation": violation,
            "feasible": feasible,
            "constraints": g,
        }

    # Evalúa todas las soluciones de una población.
    def _evaluate_population(self, population: np.ndarray) -> List[Dict[str, Any]]:
        return [self._evaluate_one(x) for x in population]

    # Inicializa una población aleatoria dentro de los límites del problema.
    def _initialize_population(self) -> np.ndarray:
        pop = self.rng.uniform(self.lb, self.ub, size=(self.population_size, self.dim))
        return np.asarray([self._clip_and_repair(x) for x in pop], dtype=float)

    # Devuelve una clave de ordenación que prioriza factibilidad, objetivo y violación de restricciones.
    def _feasibility_key(self, record: Dict[str, Any]) -> Tuple[float, float, float]:
        if record["feasible"]:
            return (0.0, float(record["objective"]), float(record["violation"]))
        return (1.0, float(record["violation"]), float(record["score"]))

    # Ordena soluciones utilizando el criterio de factibilidad.
    def _sort_by_feasibility(self, records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return sorted(records, key=self._feasibility_key)

    # Asigna rangos según el orden de factibilidad.
    def _ranks_for_records(self, records: List[Dict[str, Any]]) -> np.ndarray:
        order = sorted(range(len(records)), key=lambda i: self._feasibility_key(records[i]))
        ranks = np.empty(len(records), dtype=int)
        for rank, idx in enumerate(order, start=1):
            ranks[idx] = rank
        return ranks

    # Calcula el fitness artificial a partir del ranking.
    @staticmethod
    def _rank_fitness_from_ranks(ranks: np.ndarray) -> np.ndarray:
        n = len(ranks)
        return (n - ranks + 1).astype(float)

    # Selección por ruleta utilizando el fitness de ranking.
    # Define y documenta la función `_ranking_selection`.
    def _ranking_selection(self, records: List[Dict[str, Any]], n_select: int) -> List[Dict[str, Any]]:
        # Selecciona soluciones mediante ruleta usando el fitness de ranking.
        if n_select <= 0:
            return []
        if not records:
            return []
        ranks = self._ranks_for_records(records)
        fitness = self._rank_fitness_from_ranks(ranks)
        probs = fitness / np.sum(fitness)
        idx = self.rng.choice(len(records), size=n_select, replace=True, p=probs)
        return [records[int(i)] for i in idx]

    # Calcula el enriquecimiento relativo de una solución.
    def _rank_enrichment(self, rank: int, n: int) -> float:
        if n <= 1:
            return 1.0
        return float((n - rank) / (n - 1))

    # Calcula la probabilidad de impacto cercano a partir del enriquecimiento relativo.
    def _p_close_from_enrichment(self, enrichment: float) -> float:
        return float(self.p_min + (self.p_max - self.p_min) * np.clip(enrichment, 0.0, 1.0))

    # Calcula los límites de la vecindad local usada por el operador de impacto cercano.
    def _neighborhood_bounds(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        radius = self.neighborhood_rate * (self.ub - self.lb)
        nlb = np.maximum(self.lb, x - radius)
        nub = np.minimum(self.ub, x + radius)
        return nlb, nub

    # Operador de impacto cercano.
    # Genera una nueva solución dentro de la vecindad local de la solución actual.
    def _close_impact(self, x: np.ndarray) -> np.ndarray:
        nlb, nub = self._neighborhood_bounds(x)
        child = self.rng.uniform(nlb, nub)
        return self._clip_and_repair(child)

    # Operador de impacto distante.
    # Genera una nueva solución muestreando en todo el dominio permitido.
    def _distant_impact(self, x: np.ndarray) -> np.ndarray:
        child = self.rng.uniform(self.lb, self.ub)
        return self._clip_and_repair(child)

    # Compara dos soluciones utilizando la factibilidad.
    def _is_better(self, a: Dict[str, Any], b: Optional[Dict[str, Any]]) -> bool:
        if b is None:
            return True
        return self._feasibility_key(a) < self._feasibility_key(b)

    # Ejecuta el algoritmo NCR hasta alcanzar el presupuesto máximo de evaluaciones.
    def optimize(self, verbose: bool = False) -> NCRResult:
        population = self._initialize_population()
        pop_records = self._evaluate_population(population)

        # La mejor solución global se decide mediante ranking de factibilidad.
        # Si existe alguna solución factible, se conserva la factible con menor objetivo;
        # si no existe ninguna, se conserva la solución con menor violación total.
        best_overall: Optional[Dict[str, Any]] = None
        best_penalized: Optional[Dict[str, Any]] = None

        history = {
            "best_reported_objective": [],
            "best_penalized_score": [],
            "best_violation": [],
            "feasible_rate": [],
            "evaluations": [],
        }

        # Actualiza la mejor solución global y la mejor solución según el valor penalizado.
        def update_bests(records: List[Dict[str, Any]]) -> None:
            nonlocal best_overall, best_penalized
            for r in records:
                if self._is_better(r, best_overall):
                    best_overall = r
                if best_penalized is None or r["score"] < best_penalized["score"]:
                    best_penalized = r

        update_bests(pop_records)

        iterations = 0
        impacts_per_iteration = self.population_size * self.k

        while self.evaluations + impacts_per_iteration <= self.max_evaluations:
            # Se ordena la población actual mediante las reglas de factibilidad.
            pop_ranks = self._ranks_for_records(pop_records)
            impacts = []

            for idx, r in enumerate(pop_records):
                x = r["x"]
                rank = int(pop_ranks[idx])
                enrichment = self._rank_enrichment(rank, len(pop_records))
                p_close = self._p_close_from_enrichment(enrichment)

                for _ in range(self.k):
                    if self.rng.random() < p_close:
                        child = self._close_impact(x)
                    else:
                        child = self._distant_impact(x)
                    impacts.append(child)

            impact_records = self._evaluate_population(np.asarray(impacts, dtype=float))

            # Se construye el conjunto candidato C = P ∪ I para aplicar elitismo y selección.
            candidate_records = pop_records + impact_records
            update_bests(candidate_records)

            candidate_records_sorted = self._sort_by_feasibility(candidate_records)
            elite_count = min(int(np.floor(self.elitism_rate * self.population_size)), self.population_size)
            elites = candidate_records_sorted[:elite_count]
            selected = self._ranking_selection(candidate_records_sorted, self.population_size - elite_count)
            pop_records = elites + selected

            iterations += 1
            report = best_overall if best_overall is not None else best_penalized
            history["best_reported_objective"].append(float(report["objective"]))
            history["best_penalized_score"].append(float(best_penalized["score"]))
            history["best_violation"].append(float(report["violation"]))
            history["feasible_rate"].append(float(np.mean([r["feasible"] for r in impact_records])))
            history["evaluations"].append(float(self.evaluations))

            if verbose and (iterations % 50 == 0):
                print(
                    f"Iter={iterations:4d} | evals={self.evaluations:6d} | "
                    f"best_obj={report['objective']:.8g} | feasible={report['feasible']} | "
                    f"viol={report['violation']:.3e}"
                )

        final = best_overall if best_overall is not None else best_penalized
        if final is None:
            raise RuntimeError("No se evaluó ninguna solución.")

        return NCRResult(
            best_x=np.asarray(final["x"], dtype=float),
            best_objective=float(final["objective"]),
            best_score=float(final["score"]),
            best_violation=float(final["violation"]),
            feasible=bool(final["feasible"]),
            constraints=np.asarray(final["constraints"], dtype=float),
            evaluations=self.evaluations,
            iterations=iterations,
            history=history,
        )

## 2. Definición de los doce problemas

In [2]:
# -----------------------------------------------------------------------------
# Definiciones de problemas
# -----------------------------------------------------------------------------

# Speed Reducer

def speed_reducer_objective(x: np.ndarray) -> float:
    x1, x2, x3, x4, x5, x6, x7 = x
    return (
        0.7854 * x1 * x2**2 * (3.3333 * x3**2 + 14.9334 * x3 - 43.0934)
        + 0.7854 * (x4 * x6**2 + x5 * x7**2)
        - 1.508 * x1 * (x6**2 + x7**2)
        + 7.4777 * (x6**3 + x7**3)
    )

def speed_reducer_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2, x3, x4, x5, x6, x7 = x
    return [
        27 / (x1 * x2**2 * x3) - 1,
        397.5 / (x1 * x2**2 * x3**2) - 1,
        1.93 * x4**3 / (x2 * x3 * x6**4) - 1,
        1.93 * x5**3 / (x2 * x3 * x7**4) - 1,
        math.sqrt((745 * x4 / (x2 * x3))**2 + 16.9e6) / (110 * x6**3) - 1,
        math.sqrt((745 * x5 / (x2 * x3))**2 + 157.5e6) / (85 * x7**3) - 1,
        x2 * x3 / 40 - 1,
        5 * x2 / x1 - 1,
        x1 / (12 * x2) - 1,
        (1.5 * x6 + 1.9) / x4 - 1,
        (1.1 * x7 + 1.9) / x5 - 1,
    ]

def speed_reducer_repair(x: np.ndarray) -> np.ndarray:
    y = np.asarray(x, dtype=float).copy()
    y[2] = np.round(y[2])
    return y

# Tension-Compression Spring

def spring_objective(x: np.ndarray) -> float:
    x1, x2, x3 = x
    return (x3 + 2) * x2 * x1**2

def spring_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2, x3 = x
    return [
        1 - (x2**3 * x3) / (71785 * x1**4),
        ((4 * x2**2 - x1 * x2) / (12566 * (x2 * x1**3 - x1**4))) + 1 / (5108 * x1**2) - 1,
        1 - (140.45 * x1) / (x2**2 * x3),
        (x1 + x2) / 1.5 - 1,
    ]

def spring_repair(x: np.ndarray) -> np.ndarray:
    y = np.asarray(x, dtype=float).copy()
    y[2] = np.round(y[2])
    return y

# Pressure Vessel

def pressure_objective(x: np.ndarray) -> float:
    x1, x2, x3, x4 = x
    return 0.6224 * x1 * x3 * x4 + 1.7781 * x2 * x3**2 + 3.1661 * x1**2 * x4 + 19.84 * x1**2 * x3

def pressure_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2, x3, x4 = x
    return [
        -x1 + 0.0193 * x3,
        -x2 + 0.00954 * x3,
        -math.pi * x3**2 * x4 - (4.0 / 3.0) * math.pi * x3**3 + 1296000,
        x4 - 240,
    ]


def pressure_repair(x: np.ndarray) -> np.ndarray:
    y = np.asarray(x, dtype=float).copy()
    y[0] = np.round(y[0] / 0.0625) * 0.0625
    y[1] = np.round(y[1] / 0.0625) * 0.0625
    return y

# Welded Beam

def welded_beam_objective(x: np.ndarray) -> float:
    x1, x2, x3, x4 = x
    return 1.10471 * x1**2 * x2 + 0.04811 * x3 * x4 * (14 + x2)


def welded_beam_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2, x3, x4 = x
    P = 6000.0
    L = 14.0
    E = 30e6
    G = 12e6
    tau_max = 13600.0
    sigma_max = 30000.0
    delta_max = 0.25

    tau_prime = P / (math.sqrt(2) * x1 * x2)
    M = P * (L + x2 / 2)
    R = math.sqrt(x2**2 / 4 + ((x1 + x3) / 2) ** 2)
    J = 2 * (math.sqrt(2) * x1 * x2 * (x2**2 / 12 + ((x1 + x3) / 2) ** 2))
    tau_double = M * R / J
    tau = math.sqrt(tau_prime**2 + 2 * tau_prime * tau_double * x2 / (2 * R) + tau_double**2)
    sigma = 504000 / (x4 * x3**2)
    delta = 2.1952 / (x4 * x3**3)
    Pc = (4.013 * E * math.sqrt((x3**2 * x4**6) / 36) / (L**2)) * (1 - (x3 / (2 * L)) * math.sqrt(E / (4 * G)))

    return [
        tau - tau_max,
        sigma - sigma_max,
        x1 - x4,
        welded_beam_objective(x) - 5.0,
        0.125 - x1,
        delta - delta_max,
        P - Pc,
    ]

# Three-Bar Truss

def three_bar_objective(x: np.ndarray) -> float:
    x1, x2 = x
    l = 100.0
    return l * (2 * math.sqrt(2) * x1 + x2)


def three_bar_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2 = x
    P = 2.0
    sigma = 2.0
    den = math.sqrt(2) * x1**2 + 2 * x1 * x2
    return [
        ((math.sqrt(2) * x1 + x2) / den) * P - sigma,
        (x2 / den) * P - sigma,
        (1 / (math.sqrt(2) * x2 + x1)) * P - sigma,
    ]

# Multiple Disk Clutch Brake

def clutch_objective(x: np.ndarray) -> float:
    ri, ro, t, F, Z = x
    rho = 7.8e-6
    return math.pi * (ro**2 - ri**2) * t * (Z + 1) * rho


def clutch_constraints(x: np.ndarray) -> Sequence[float]:
    ri, ro, t, F, Z = x
    delta_r = 20.0
    Lmax = 30.0
    mu = 0.6
    Vsr_max = 10.0
    delta = 0.5
    s = 1.5
    Tmax = 15.0
    n = 250.0
    Iz = 55.0
    Ms = 40.0
    Mf = 3.0
    Pmax = 1.0

    A = math.pi * (ro**2 - ri**2)
    Prz = F / A
    Rsr = (2.0 / 3.0) * (ro**3 - ri**3) / (ro**2 - ri**2)
    Vsr = math.pi * Rsr * n / 30.0 / 1000.0  # aproximación en m/s
    Mh = (2.0 / 3.0) * mu * F * Z * (ro**3 - ri**3) / (ro**2 - ri**2)
    omega = math.pi * n / 30.0
    T = Iz * omega / max(Mh + Mf, EPS)

    return [
        delta_r + ri - ro,
        (Z + 1) * (t + delta) - Lmax,
        Prz - Pmax,
        Prz * Vsr - Pmax * Vsr_max,
        Vsr - Vsr_max,
        T - Tmax,
        s * Ms - Mh,
        -T,
    ]

def clutch_repair(x: np.ndarray) -> np.ndarray:
    y = np.asarray(x, dtype=float).copy()
    y[4] = np.round(y[4])
    return y

# Himmelblau

def himmelblau_objective(x: np.ndarray) -> float:
    x1, x2, x3, x4, x5 = x
    return 5.3578547 * x3**2 + 0.8356891 * x1 * x5 + 37.293239 * x1 - 40792.141


def himmelblau_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2, x3, x4, x5 = x
    G1 = 85.334407 + 0.0056858 * x2 * x5 + 0.0006262 * x1 * x4 - 0.0022053 * x3 * x5
    G2 = 80.51249 + 0.0071317 * x2 * x5 + 0.0029955 * x1 * x2 - 0.0021813 * x3**2
    G3 = 9.300961 + 0.0047026 * x3 * x5 + 0.00125447 * x1 * x3 - 0.0019085 * x3 * x4
    return [-G1, G1 - 92, 90 - G2, G2 - 110, 20 - G3, G3 - 25]

# Cantilever Beam

def cantilever_objective(x: np.ndarray) -> float:
    return 0.6224 * float(np.sum(x))

def cantilever_constraints(x: np.ndarray) -> Sequence[float]:
    x1, x2, x3, x4, x5 = x
    return [60 / x1**3 + 27 / x2**3 + 19 / x3**3 + 7 / x4**3 + 1 / x5**3 - 1]

# Tubular Column

def tubular_objective(x: np.ndarray) -> float:
    d, t = x
    return 9.8 * d * t + 2 * d

def tubular_constraints(x: np.ndarray) -> Sequence[float]:
    d, t = x
    P = 2500.0
    E = 0.85e6
    sigma_y = 500.0
    L = 250.0
    return [
        P / (math.pi * d * t * sigma_y) - 1,
        8 * P * L**2 / (math.pi**3 * E * d * t * (d**2 + t**2)) - 1,
        2.0 / d - 1,
        d / 14.0 - 1,
        0.2 / t - 1,
        t / 0.8 - 1,
    ]

# Piston Lever

def piston_objective(x: np.ndarray) -> float:
    H, B, D, X = x
    L1 = math.sqrt((X - B) ** 2 + H**2)
    L2 = math.sqrt((X * math.sin(math.pi / 4) + H) ** 2 + (B - X * math.cos(math.pi / 4)) ** 2)
    return 0.25 * math.pi * D**2 * (L2 - L1)

def piston_constraints(x: np.ndarray) -> Sequence[float]:
    H, B, D, X = x
    theta = math.pi / 4
    Q = 10000.0
    L = 240.0
    Mmax = 1.8e6
    pressure = 1500.0
    F = math.pi * pressure * D**2 / 4
    L1 = math.sqrt((X - B) ** 2 + H**2)
    L2 = math.sqrt((X * math.sin(theta) + H) ** 2 + (B - X * math.cos(theta)) ** 2)
    R = abs(-X * (X * math.sin(theta) + H) + H * (B - X * math.cos(theta))) / max(L1, EPS)
    return [
        Q * L * math.cos(theta) - R * F,
        Q * (L - X) - Mmax,
        1.2 * (L2 - L1) - L1,
        D / 2 - B,
    ]

# Robot Gripper

def _robot_geometry(x: np.ndarray, z: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    a, b, c, e, f, l, delta = x
    g = np.sqrt(e**2 + (z - l) ** 2)
    theta = np.arctan2(e, l - z)
    arg_alpha = (a**2 + g**2 - b**2) / np.maximum(2 * a * g, EPS)
    arg_beta = (b**2 + g**2 - a**2) / np.maximum(2 * b * g, EPS)
    arg_alpha = np.clip(arg_alpha, -1.0, 1.0)
    arg_beta = np.clip(arg_beta, -1.0, 1.0)
    alpha = np.arccos(arg_alpha) + theta
    beta = np.arccos(arg_beta) - theta
    y = 2 * (f + e + c * np.sin(beta + delta))
    return alpha, beta, y

def robot_gripper_objective(x: np.ndarray) -> float:
    z = np.linspace(0.0, 100.0, ROBOT_Z_POINTS)
    a, b, c, e, f, l, delta = x
    alpha, beta, _ = _robot_geometry(x, z)
    Fk = (b * 100.0 * np.sin(alpha + beta)) / np.maximum(2 * c * np.cos(alpha), EPS)
    Fk = Fk[np.isfinite(Fk)]
    if Fk.size == 0:
        return INVALID_SCORE
    return float(abs(np.max(Fk) - np.min(Fk)))


def robot_gripper_constraints(x: np.ndarray) -> Sequence[float]:
    a, b, c, e, f, l, delta = x
    zmax = 100.0
    Ymin = 50.0
    Ymax = 100.0
    YG = 150.0
    _, _, yzmax_arr = _robot_geometry(x, np.array([zmax]))
    _, _, y0_arr = _robot_geometry(x, np.array([0.0]))
    yzmax = float(yzmax_arr[0])
    y0 = float(y0_arr[0])
    return [
        yzmax - Ymin,
        -yzmax,
        Ymax - y0,
        y0 - YG,
        l**2 + e**2 - (a + b) ** 2,
        b**2 - (a - e) ** 2 - (l - zmax) ** 2,
        zmax - l,
    ]

# Corrugated Bulkhead

def corrugated_objective(x: np.ndarray) -> float:
    b, h, l, t = x
    root = math.sqrt(max(l**2 - h**2, EPS))
    denom = b + root
    if denom <= EPS:
        return INVALID_SCORE
    return 5.885 * t * (b + l) / denom

def corrugated_constraints(x: np.ndarray) -> Sequence[float]:
    b, h, l, t = x
    root = math.sqrt(max(l**2 - h**2, EPS))
    c1 = t * h * (0.4 * b + 1 / 6) - 8.94 * (b + root)
    c2 = t * h**2 * (0.2 * b + 1 / 12) - 2.2 * (8.94 * (b + root)) ** (4 / 3)
    c3 = t - 0.0156 * b - 0.15
    c4 = t - 0.0156 * l - 0.15
    c5 = t - 1.15
    c6 = l - h
    # La formulación original usa c_i >= 0; se transforma a g_i <= 0.
    return [-c1, -c2, -c3, -c4, -c5, -c6]


ROBOT_Z_POINTS = 51

# Crea las instancias de los problemas con objetivo, restricciones y reparación.

# Construye y devuelve la lista de los doce problemas.
def get_benchmark_problems() -> List[BenchmarkProblem]:
    return [
        BenchmarkProblem(
            name="Speed Reducer",
            bounds=[(2.6, 3.6), (0.7, 0.8), (17, 28), (7.3, 8.3), (7.3, 8.3), (2.9, 3.9), (5.0, 5.5)],
            objective=speed_reducer_objective,
            constraints=speed_reducer_constraints,
            repair=speed_reducer_repair,
            known_best=2.9944244658e3,
        ),
        BenchmarkProblem(
            name="Tension-Compression Spring",
            bounds=[(0.05, 2.0), (0.25, 1.30), (2.0, 15.0)],
            objective=spring_objective,
            constraints=spring_constraints,
            repair=spring_repair,
            known_best=1.2665232788e-2,
        ),
        BenchmarkProblem(
            name="Pressure Vessel",
            bounds=[(0.0625, 99 * 0.0625), (0.0625, 99 * 0.0625), (10.0, 200.0), (10.0, 200.0)],
            objective=pressure_objective,
            constraints=pressure_constraints,
            repair=pressure_repair,
            known_best=5.8853327736e3,
        ),
        BenchmarkProblem(
            name="Welded Beam",
            bounds=[(0.125, 2.0), (0.1, 10.0), (0.1, 10.0), (0.1, 2.0)],
            objective=welded_beam_objective,
            constraints=welded_beam_constraints,
            known_best=1.6702177263,
        ),
        BenchmarkProblem(
            name="Three-Bar Truss",
            bounds=[(1e-6, 1.0), (1e-6, 1.0)],
            objective=three_bar_objective,
            constraints=three_bar_constraints,
            known_best=2.6389584338e2,
        ),
        BenchmarkProblem(
            name="Multiple Disk Clutch Brake",
            bounds=[(60.0, 80.0), (90.0, 110.0), (1.0, 3.0), (0.0, 1000.0), (2.0, 9.0)],
            objective=clutch_objective,
            constraints=clutch_constraints,
            repair=clutch_repair,
            known_best=2.3524245790e-1,
        ),
        BenchmarkProblem(
            name="Himmelblau Function",
            bounds=[(78.0, 102.0), (33.0, 45.0), (27.0, 45.0), (27.0, 45.0), (27.0, 45.0)],
            objective=himmelblau_objective,
            constraints=himmelblau_constraints,
            known_best=-3.0665538672e4,
        ),
        BenchmarkProblem(
            name="Cantilever Beam",
            bounds=[(0.01, 100.0)] * 5,
            objective=cantilever_objective,
            constraints=cantilever_constraints,
            known_best=1.3399576,
        ),
        BenchmarkProblem(
            name="Tubular Column",
            bounds=[(2.0, 14.0), (0.2, 0.8)],
            objective=tubular_objective,
            constraints=tubular_constraints,
            known_best=26.486361473,
        ),
        BenchmarkProblem(
            name="Piston Lever",
            bounds=[(0.05, 500.0), (0.05, 500.0), (0.05, 500.0), (0.05, 120.0)],
            objective=piston_objective,
            constraints=piston_constraints,
            known_best=8.41269832311,
        ),
        BenchmarkProblem(
            name="Robot Gripper",
            bounds=[(10.0, 150.0), (10.0, 150.0), (100.0, 200.0), (0.0, 50.0), (10.0, 150.0), (100.0, 300.0), (1.0, 3.14)],
            objective=robot_gripper_objective,
            constraints=robot_gripper_constraints,
            known_best=2.5287918415,
        ),
        BenchmarkProblem(
            name="Corrugated Bulkhead",
            bounds=[(0.0, 100.0), (0.0, 100.0), (0.0, 100.0), (0.0, 5.0)],
            objective=corrugated_objective,
            constraints=corrugated_constraints,
            known_best=6.8429580100808,
        ),
    ]

## 3. Parámetros generales y funciones de ejecución

In [3]:
DEFAULT_NCR_PARAMS: Dict[str, Any] = {
    "population_size": 30,
    "k": 3,
    "p_min": 0.20,
    "p_max": 0.90,
    "elitism_rate": 0.10,
    "neighborhood_rate": 0.10,
    "penalty_factor": 1e12,
    "max_evaluations": 30000,
}

# Diccionario opcional para fijar parámetros específicos por problema.
# Si un parámetro no aparece aquí, se hereda de DEFAULT_NCR_PARAMS.
PARAMS_BY_PROBLEM: Dict[str, Dict[str, Any]] = {}


def run_problem(
    problem: BenchmarkProblem,
    n_runs: int = 30,
    base_params: Optional[Dict[str, Any]] = None,
    params_by_problem: Optional[Dict[str, Dict[str, Any]]] = None,
    seed_start: int = 0,
    verbose: bool = False,
) -> List[Dict[str, Any]]:
    # Ejecuta NCR varias veces sobre un problema concreto y guarda los resultados individuales.
    base_params = dict(DEFAULT_NCR_PARAMS if base_params is None else base_params)
    params_by_problem = {} if params_by_problem is None else params_by_problem
    params = dict(base_params)
    params.update(params_by_problem.get(problem.name, {}))

    rows: List[Dict[str, Any]] = []
    for run in range(n_runs):
        seed = seed_start + run
        opt = NCR(
            objective=problem.objective,
            constraints=problem.constraints,
            repair=problem.repair,
            bounds=problem.bounds,
            seed=seed,
            **params,
        )
        start = time.time()
        result = opt.optimize(verbose=False)
        elapsed = time.time() - start

        row = {
            "Problem": problem.name,
            "Run": run + 1,
            "Seed": seed,
            "Objective": result.best_objective,
            "Score": result.best_score,
            "Violation": result.best_violation,
            "Feasible": result.feasible,
            "Evaluations": result.evaluations,
            "Iterations": result.iterations,
            "ElapsedSeconds": elapsed,
            "BestX": ";".join(f"{v:.9f}" for v in result.best_x),
            "KnownBest": problem.known_best,
            **{f"param_{k}": v for k, v in params.items()},
        }
        rows.append(row)
        if verbose:
            print(
                f"{problem.name:32s} | run {run + 1:02d}/{n_runs} | "
                f"obj={result.best_objective:.9f} | feasible={result.feasible} | "
                f"viol={result.best_violation:.3e} | evals={result.evaluations}"
            )
    return rows


# Resume las ejecuciones independientes calculando Best, Mean, Worst, SD y factibilidad.
def summarize_runs(runs_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for problem, group in runs_df.groupby("Problem", sort=False):
        values = group["Objective"].astype(float).to_numpy()
        feasible_values = group.loc[group["Feasible"].astype(bool), "Objective"].astype(float).to_numpy()
        stat_values = feasible_values if feasible_values.size > 0 else values
        best_idx = group["Objective"].astype(float).idxmin()

        rows.append({
            "Problem": problem,
            "Best": float(np.min(stat_values)),
            "Mean": float(np.mean(stat_values)),
            "Worst": float(np.max(stat_values)),
            "SD": float(np.std(stat_values, ddof=1)) if stat_values.size > 1 else 0.0,
            "FeasibleRuns": int(group["Feasible"].sum()),
            "Runs": int(len(group)),
            "KnownBest": float(group["KnownBest"].iloc[0]) if pd.notna(group["KnownBest"].iloc[0]) else np.nan,
            "BestX": group.loc[best_idx, "BestX"],
        })
    return pd.DataFrame(rows)


# Ejecuta NCR sobre todos los benchmarks y guarda las tablas de resultados.
def run_all_benchmarks(
    n_runs: int = 30,
    params: Optional[Dict[str, Any]] = None,
    params_by_problem: Optional[Dict[str, Dict[str, Any]]] = None,
    seed_start: int = 0,
    verbose: bool = True,
    save_csv: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    problems = get_benchmark_problems()
    all_rows: List[Dict[str, Any]] = []
    for p_idx, problem in enumerate(problems):
        if verbose:
            print(f"\n=== {p_idx + 1}/{len(problems)}: {problem.name} ===")
        rows = run_problem(
            problem=problem,
            n_runs=n_runs,
            base_params=params,
            params_by_problem=params_by_problem,
            seed_start=seed_start + p_idx * 1000,
            verbose=verbose,
        )
        all_rows.extend(rows)

    runs_df = pd.DataFrame(all_rows)
    summary_df = summarize_runs(runs_df)

    if save_csv:
        runs_df.to_csv("ncr_final_runs.csv", index=False, float_format="%.9f")
        summary_df.to_csv("ncr_final_summary.csv", index=False, float_format="%.9f")

    return summary_df, runs_df

## 4. Selección recomendada de parámetros

In [4]:
# -----------------------------------------------------------------------------
# Selección de parámetros
# -----------------------------------------------------------------------------

PARAMETER_SEARCH_SPACE: Dict[str, List[Any]] = {
    "population_size": [20, 30, 50],
    "k": [2, 3, 4, 5],
    "p_min": [0.05, 0.10, 0.20, 0.30],
    "p_max": [0.70, 0.80, 0.90, 1.00],
    "elitism_rate": [0.0, 0.05, 0.10, 0.20],
    "neighborhood_rate": [0.02, 0.05, 0.10, 0.20],
    "penalty_factor": [1e6, 1e8, 1e10, 1e12]
}

INTEGER_PARAM_NAMES = {"population_size", "k", "max_evaluations"}
TUNABLE_PARAM_NAMES = [
    "population_size",
    "k",
    "p_min",
    "p_max",
    "elitism_rate",
    "neighborhood_rate",
    "penalty_factor",
]


# Convierte parámetros enteros y comprueba que p_min sea menor que p_max.
def _sanitize_params(params: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(params)
    for name in INTEGER_PARAM_NAMES:
        if name in out:
            out[name] = int(out[name])
    if "p_min" in out and "p_max" in out and not (float(out["p_min"]) < float(out["p_max"])):
        raise ValueError(f"Probabilidades no válidas: p_min={out['p_min']} debe ser menor que p_max={out['p_max']}")
    return out


# Define y documenta la función `sample_parameter_configuration`.
def sample_parameter_configuration(
    search_space: Dict[str, Sequence[Any]],
    rng: np.random.Generator,
) -> Dict[str, Any]:
    # Genera aleatoriamente una configuración de parámetros válida.
    for _ in range(1000):
        config: Dict[str, Any] = {}
        for name, values in search_space.items():
            value = rng.choice(list(values))
            if hasattr(value, "item"):
                value = value.item()
            config[name] = value
        try:
            return _sanitize_params(config)
        except ValueError:
            continue
    raise RuntimeError("No se pudo generar una configuración válida. Revisar los valores de p_min y p_max.")


# Crea una representación hashable para detectar configuraciones duplicadas.
def _configuration_signature(config: Dict[str, Any]) -> Tuple[Tuple[str, Any], ...]:
    return tuple(sorted((k, float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v) for k, v in config.items()))



def generate_unique_parameter_configurations(
    search_space: Dict[str, Sequence[Any]],
    n_configs: int,
    seed: int = 123,
) -> List[Dict[str, Any]]:
    # Generar configuraciones aleatorias únicas para la fase de selección de parámetros
    rng = np.random.default_rng(seed)
    configs: List[Dict[str, Any]] = []
    seen = set()
    attempts = 0
    max_attempts = max(1000, n_configs * 100)
    while len(configs) < n_configs and attempts < max_attempts:
        attempts += 1
        c = sample_parameter_configuration(search_space, rng)
        sig = _configuration_signature(c)
        if sig not in seen:
            seen.add(sig)
            configs.append(c)
    if len(configs) < n_configs:
        print(f"Aviso: se solicitaron {n_configs} configuraciones, pero solo se generaron {len(configs)} únicas.")
    return configs


# Define el criterio de ordenación usado para comparar configuraciones de parámetros.
def _tuning_score_tuple(row: Dict[str, Any]) -> Tuple[Any, ...]:
    return (
        -int(row["FeasibleRuns"]),
        float(row["MeanObjective"]),
        float(row["BestObjective"]),
        float(row["SDObjective"]),
        float(row["MeanViolation"]),
    )


# Evalúa una configuración de parámetros mediante varias ejecuciones del algoritmo.
def evaluate_parameter_configuration(
    problem: BenchmarkProblem,
    config: Dict[str, Any],
    base_params: Optional[Dict[str, Any]] = None,
    n_runs: int = 3,
    seed_start: int = 0,
    max_evaluations: Optional[int] = None,
    phase: str = "inicial",
    config_id: int = 0,
    verbose: bool = False,
) -> Dict[str, Any]:
    params = dict(DEFAULT_NCR_PARAMS if base_params is None else base_params)
    params.update(config)
    if max_evaluations is not None:
        params["max_evaluations"] = int(max_evaluations)
    params = _sanitize_params(params)

    run_rows = run_problem(
        problem=problem,
        n_runs=n_runs,
        base_params=params,
        params_by_problem=None,
        seed_start=seed_start,
        verbose=False,
    )
    df = pd.DataFrame(run_rows)
    objectives = df["Objective"].astype(float).to_numpy()
    violations = df["Violation"].astype(float).to_numpy()
    feasible_mask = df["Feasible"].astype(bool).to_numpy()
    feasible_objectives = objectives[feasible_mask]
    stat_objectives = feasible_objectives if feasible_objectives.size > 0 else objectives

    row: Dict[str, Any] = {
        "Problem": problem.name,
        "Phase": phase,
        "ConfigID": int(config_id),
        "Runs": int(n_runs),
        "FeasibleRuns": int(np.sum(feasible_mask)),
        "FeasibilityRate": float(np.mean(feasible_mask)),
        "BestObjective": float(np.min(stat_objectives)),
        "MeanObjective": float(np.mean(stat_objectives)),
        "WorstObjective": float(np.max(stat_objectives)),
        "SDObjective": float(np.std(stat_objectives, ddof=1)) if stat_objectives.size > 1 else 0.0,
        "MeanViolation": float(np.mean(violations)),
        "MaxEvaluations": int(params["max_evaluations"]),
        "SeedStart": int(seed_start),
    }
    row.update({f"param_{k}": params[k] for k in TUNABLE_PARAM_NAMES if k in params})
    row["RankTuple"] = _tuning_score_tuple(row)

    if verbose:
        print(
            f"{problem.name:32s} | {phase:7s} | cfg={config_id:03d} | "
            f"feas={row['FeasibleRuns']}/{n_runs} | mean={row['MeanObjective']:.8g} | "
            f"best={row['BestObjective']:.8g} | viol={row['MeanViolation']:.3e}"
        )
    return row


# Realiza la selección de parámetros de NCR para un problema
def tune_problem_parameters(
    problem: BenchmarkProblem,
    search_space: Optional[Dict[str, Sequence[Any]]] = None,
    base_params: Optional[Dict[str, Any]] = None,
    n_initial_configs: int = 25,
    initial_runs: int = 3,
    initial_max_evaluations: int = 5000,
    top_configs: int = 5,
    refine_runs: int = 10,
    refine_max_evaluations: int = 15000,
    seed: int = 123,
    verbose: bool = True,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    search_space = PARAMETER_SEARCH_SPACE if search_space is None else dict(search_space)
    base_params = dict(DEFAULT_NCR_PARAMS if base_params is None else base_params)

    configs = generate_unique_parameter_configurations(
        search_space=search_space,
        n_configs=n_initial_configs,
        seed=seed,
    )

    rows: List[Dict[str, Any]] = []
    if verbose:
        print(f"\n--- Ajuste de {problem.name}: exploración inicial ---")
    for i, config in enumerate(configs):
        row = evaluate_parameter_configuration(
            problem=problem,
            config=config,
            base_params=base_params,
            n_runs=initial_runs,
            seed_start=seed * 100000 + i * 100,
            max_evaluations=initial_max_evaluations,
            phase="inicial",
            config_id=i,
            verbose=verbose,
        )
        rows.append(row)

    initial_df = pd.DataFrame(rows)
    initial_sorted = initial_df.sort_values(by=["FeasibleRuns", "MeanObjective", "BestObjective", "SDObjective", "MeanViolation"],
                                            ascending=[False, True, True, True, True])
    selected_initial = initial_sorted.head(top_configs)

    if verbose:
        print(f"\n--- Ajuste de {problem.name}: refinamiento ---")
    refine_rows: List[Dict[str, Any]] = []
    for j, (_, init_row) in enumerate(selected_initial.iterrows()):
        config = {name: init_row[f"param_{name}"] for name in TUNABLE_PARAM_NAMES if f"param_{name}" in init_row}
        config = _sanitize_params(config)
        original_config_id = int(init_row["ConfigID"])
        row = evaluate_parameter_configuration(
            problem=problem,
            config=config,
            base_params=base_params,
            n_runs=refine_runs,
            seed_start=seed * 100000 + 50000 + j * 100,
            max_evaluations=refine_max_evaluations,
            phase="refinamiento",
            config_id=original_config_id,
            verbose=verbose,
        )
        refine_rows.append(row)

    refine_df = pd.DataFrame(refine_rows)
    all_df = pd.concat([initial_df, refine_df], ignore_index=True)
    refined_sorted = refine_df.sort_values(by=["FeasibleRuns", "MeanObjective", "BestObjective", "SDObjective", "MeanViolation"],
                                           ascending=[False, True, True, True, True])
    best = refined_sorted.iloc[0]
    selected_params = dict(base_params)
    for name in TUNABLE_PARAM_NAMES:
        if f"param_{name}" in best:
            selected_params[name] = best[f"param_{name}"]
    selected_params = _sanitize_params(selected_params)
    # La evaluación final usa el presupuesto completo, no el presupuesto reducido del tuning.
    selected_params["max_evaluations"] = int(base_params.get("max_evaluations", DEFAULT_NCR_PARAMS["max_evaluations"]))

    if verbose:
        print(f"Seleccionado para {problem.name}: " + ", ".join(f"{k}={selected_params[k]}" for k in TUNABLE_PARAM_NAMES))

    return selected_params, all_df


# Ejecuta la selección de parámetros para todos los problemas benchmark.
def tune_all_problems(
    problems: Optional[List[BenchmarkProblem]] = None,
    search_space: Optional[Dict[str, Sequence[Any]]] = None,
    base_params: Optional[Dict[str, Any]] = None,
    n_initial_configs: int = 25,
    initial_runs: int = 3,
    initial_max_evaluations: int = 5000,
    top_configs: int = 5,
    refine_runs: int = 10,
    refine_max_evaluations: int = 15000,
    seed: int = 123,
    verbose: bool = True,
    save_csv: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Dict[str, Any]]]:
    problems = get_benchmark_problems() if problems is None else problems
    base_params = dict(DEFAULT_NCR_PARAMS if base_params is None else base_params)

    all_tuning_rows: List[pd.DataFrame] = []
    selected_rows: List[Dict[str, Any]] = []
    params_by_problem: Dict[str, Dict[str, Any]] = {}

    for p_idx, problem in enumerate(problems):
        selected_params, tuning_df = tune_problem_parameters(
            problem=problem,
            search_space=search_space,
            base_params=base_params,
            n_initial_configs=n_initial_configs,
            initial_runs=initial_runs,
            initial_max_evaluations=initial_max_evaluations,
            top_configs=top_configs,
            refine_runs=refine_runs,
            refine_max_evaluations=refine_max_evaluations,
            seed=seed + p_idx * 1000,
            verbose=verbose,
        )
        all_tuning_rows.append(tuning_df)

        variable_params = {k: selected_params[k] for k in TUNABLE_PARAM_NAMES if k in selected_params}
        params_by_problem[problem.name] = variable_params
        selected_rows.append({
            "Problem": problem.name,
            **{k: variable_params.get(k, np.nan) for k in TUNABLE_PARAM_NAMES},
            "max_evaluations_final": int(base_params.get("max_evaluations", DEFAULT_NCR_PARAMS["max_evaluations"])),
            "alpha": "no_usado",
            "alpha_note": "sustituido por p_min y p_max",
        })

    selected_params_df = pd.DataFrame(selected_rows)
    tuning_results_df = pd.concat(all_tuning_rows, ignore_index=True) if all_tuning_rows else pd.DataFrame()

    if save_csv:
        selected_params_df.to_csv("ncr_selected_params_by_problem.csv", index=False, float_format="%.9f")
        tuning_results_df.to_csv("ncr_tuning_all_configs.csv", index=False, float_format="%.9f")
        write_latex_parameter_table(selected_params_df, "ncr_selected_params_latex_table.tex")

    return selected_params_df, tuning_results_df, params_by_problem


# Convierte la tabla de parámetros seleccionados en un diccionario por problema.
def params_dataframe_to_dict(params_df: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
    params_by_problem: Dict[str, Dict[str, Any]] = {}
    for _, row in params_df.iterrows():
        problem = str(row["Problem"])
        params: Dict[str, Any] = {}
        for name in TUNABLE_PARAM_NAMES:
            if name in row and pd.notna(row[name]):
                params[name] = row[name]
        params_by_problem[problem] = _sanitize_params(params)
    return params_by_problem


# Genera una tabla con los parámetros seleccionados para cada problema
def write_latex_parameter_table(params_df: pd.DataFrame, path: str = "ncr_selected_params_latex_table.tex") -> None:
    lines = [
        r"\begin{table}[H]",
        r"\centering",
        r"\caption{Parámetros de NCR seleccionados para cada benchmark}",
        r"\label{tab:parametros_ncr_benchmark}",
        r"\begin{tabular}{lrrrrrrr}",
        r"\hline",
        r"\textbf{Problema} & $\mathbf{F}$ & $\mathbf{k}$ & $\mathbf{p_{\min}}$ & $\mathbf{p_{\max}}$ & $\mathbf{e}$ & $\boldsymbol{\rho}$ & $\boldsymbol{\lambda}$ \\",
        r"\hline",
    ]
    for _, row in params_df.iterrows():
        problem = str(row["Problem"])
        F = int(row["population_size"])
        k = int(row["k"])
        p_min = float(row["p_min"])
        p_max = float(row["p_max"])
        e = float(row["elitism_rate"])
        rho = float(row["neighborhood_rate"])
        lam = float(row["penalty_factor"])
        lines.append(f"{problem} & {F} & {k} & {p_min:.2f} & {p_max:.2f} & {e:.2f} & {rho:.2f} & {lam:.0e} " + "\\\\")
    lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "",
    ])
    Path(path).write_text("\n".join(lines), encoding="utf-8")


# Ejecuta la selección de parámetros y después la evaluación final completa.
def run_complete_tuning_and_final_experiment(
    n_initial_configs: int = 25,
    initial_runs: int = 3,
    initial_max_evaluations: int = 5000,
    top_configs: int = 5,
    refine_runs: int = 10,
    refine_max_evaluations: int = 15000,
    final_runs: int = 30,
    seed: int = 123,
    seed_start_final: int = 0,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    selected_params_df, tuning_df, params_by_problem = tune_all_problems(
        n_initial_configs=n_initial_configs,
        initial_runs=initial_runs,
        initial_max_evaluations=initial_max_evaluations,
        top_configs=top_configs,
        refine_runs=refine_runs,
        refine_max_evaluations=refine_max_evaluations,
        seed=seed,
        verbose=verbose,
        save_csv=True,
    )
    summary_df, runs_df = run_all_benchmarks(
        n_runs=final_runs,
        params=DEFAULT_NCR_PARAMS,
        params_by_problem=params_by_problem,
        seed_start=seed_start_final,
        verbose=verbose,
        save_csv=True,
    )
    return selected_params_df, tuning_df, summary_df, runs_df

## 5. Tabla final y configuración recomendada

In [5]:
# Muestra una tabla final con todas las magnitudes obtenidas
def mostrar_tabla_resultados(summary_df: pd.DataFrame) -> None:
    columnas = [
        "Problem",
        "Best",
        "Mean",
        "Worst",
        "SD",
        "FeasibleRuns",
        "Runs",
        "KnownBest",
        "BestX",
    ]
    tabla = summary_df.loc[:, [columna for columna in columnas if columna in summary_df.columns]].copy()

    print("\n=== TABLA FINAL DE RESULTADOS ===")
    print(
        tabla.to_string(
            index=False,
            float_format=lambda valor: f"{valor:.9f}",
        )
    )


# Ejecuta el ajuste de parámetros y la evaluación final según el modo seleccionado.
def main() -> None:
    run_mode = "recommended"

    if run_mode == "quick_test":
        tuning_config = {
            "n_initial_configs": 3,
            "initial_runs": 1,
            "initial_max_evaluations": 500,
            "top_configs": 1,
            "refine_runs": 1,
            "refine_max_evaluations": 1000,
            "seed": 123,
            "save_csv": True,
            "verbose": True,
        }
        final_runs = 2
    elif run_mode == "recommended":
        tuning_config = {
            "n_initial_configs": 25,
            "initial_runs": 3,
            "initial_max_evaluations": 5000,
            "top_configs": 5,
            "refine_runs": 10,
            "refine_max_evaluations": 15000,
            "seed": 123,
            "save_csv": True,
            "verbose": True,
        }
        final_runs = 30
    else:
        raise ValueError('run_mode debe ser "quick_test" o "recommended".')

    _, _, params_by_problem = tune_all_problems(**tuning_config)

    summary_df, _ = run_all_benchmarks(
        n_runs=final_runs,
        params=DEFAULT_NCR_PARAMS,
        params_by_problem=params_by_problem,
        seed_start=0,
        verbose=True,
        save_csv=True,
    )

    mostrar_tabla_resultados(summary_df)

## 6. Ejecución completa

Esta celda realiza directamente la configuración **recommended**:

- 25 configuraciones iniciales por problema.
- 3 ejecuciones en la exploración inicial.
- Refinamiento de las 5 mejores configuraciones con 10 ejecuciones.
- 30 ejecuciones finales independientes por problema.
- Generación de la tabla final con 9 decimales.

In [ ]:
# Ejecuta el ajuste recomendado y el experimento final completo.
main()